# 🔄 Taller de ETL con Python en Google Colab
## Curso de Ingeniería de Datos

En este notebook vamos a construir, paso a paso, un proceso **ETL** completo:

- **E**xtract (Extracción): obtener datos desde distintas fuentes
- **T**ransform (Transformación): limpiar, validar y enriquecer los datos
- **L**oad (Carga): almacenar los datos transformados en un destino final (un *data warehouse* simulado con SQLite)

**Caso de uso:** Somos el equipo de datos de una cadena de tiendas y necesitamos consolidar las ventas diarias que llegan "sucias" desde el sistema de punto de venta, junto con el catálogo de tiendas, para generar un reporte limpio y agregado.

> 💡 Todo el ejercicio funciona sin conexión a internet ni archivos externos: los datos "sucios" se generan dentro del propio notebook para que sea 100% reproducible en cualquier cuenta de Colab.

## 🎯 Objetivos de aprendizaje

Al terminar este notebook deberías poder:

1. Explicar la diferencia entre extracción, transformación y carga.
2. Identificar problemas típicos de calidad de datos (nulos, duplicados, tipos incorrectos, valores inválidos).
3. Implementar limpiezas y transformaciones con `pandas`.
4. Cargar datos limpios en una base de datos (SQLite) y en archivos (CSV/Parquet).
5. Escribir consultas SQL sobre el resultado final del pipeline.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime, timedelta

# Fijamos la semilla para que el ejercicio sea reproducible para todos los estudiantes
np.random.seed(42)

print("Librerías cargadas correctamente ✅")
print(f"Versión de pandas: {pd.__version__}")

---
## 1️⃣ EXTRACT — Extracción de datos

En un caso real, los datos suelen venir de:
- Un sistema transaccional (la base de datos del punto de venta)
- Un archivo plano exportado por otro sistema (CSV, Excel)
- Una API externa (por ejemplo, tasas de cambio, clima, etc.)

Para este ejercicio **simulamos dos fuentes**:

1. `ventas_raw.csv` → transacciones de venta (con errores típicos a propósito)
2. Un `DataFrame` con el catálogo de tiendas (como si viniera de otra tabla o sistema)

### 1.1 Generamos la fuente "sucia" de ventas (simula el sistema de origen)

In [ ]:
n = 500
tiendas_ids = [101, 102, 103, 104, None]  # None simula un error de captura en la tienda

fechas_base = [datetime(2025, 1, 1) + timedelta(days=int(x)) for x in np.random.randint(0, 30, n)]

ventas_sucias = pd.DataFrame({
    "id_venta": range(1, n + 1),
    "id_tienda": np.random.choice(tiendas_ids, n),
    "producto": np.random.choice(
        ["Laptop", "laptop", "MOUSE", "Mouse", "Teclado", "teclado ", "Monitor", None],
        n
    ),
    "cantidad": np.random.choice([1, 2, 3, 4, 5, -1, 0], n),          # -1 y 0 son errores de captura
    "precio_unitario": np.random.choice([250000, 45000, 60000, 380000, np.nan], n),
    "fecha": fechas_base,
})

# Insertamos filas duplicadas a propósito, como suele pasar en integraciones reales
ventas_sucias = pd.concat([ventas_sucias, ventas_sucias.sample(15, random_state=1)], ignore_index=True)

ventas_sucias.to_csv("ventas_raw.csv", index=False)
print(f"Archivo 'ventas_raw.csv' generado con {len(ventas_sucias)} filas (incluye errores a propósito)")
ventas_sucias.head(10)

### 1.2 Extraemos (leemos) la fuente de ventas

Esto simula el paso de **Extract**: leer tal cual llega el archivo, sin corregir nada todavía.

In [ ]:
df_ventas = pd.read_csv("ventas_raw.csv", parse_dates=["fecha"])
print(f"Filas extraídas: {len(df_ventas)}")
df_ventas.info()

### 1.3 Extraemos el catálogo de tiendas (segunda fuente)

En la práctica esto podría venir de otra base de datos, una API interna o un archivo maestro.

In [ ]:
df_tiendas = pd.DataFrame({
    "id_tienda": [101, 102, 103, 104],
    "nombre_tienda": ["Bogotá Centro", "Medellín Poblado", "Cali Norte", "Barranquilla Mall"],
    "region": ["Andina", "Andina", "Pacífico", "Caribe"],
})
df_tiendas

---
## 2️⃣ TRANSFORM — Transformación y limpieza

Antes de transformar es clave **perfilar los datos**: entender qué tan "sucios" están y qué problemas hay que corregir.

In [ ]:
print("Valores nulos por columna:")
print(df_ventas.isna().sum())

print("\nFilas duplicadas exactas:", df_ventas.duplicated().sum())
print("\nValores únicos en 'producto':", df_ventas["producto"].unique())
print("\nValores únicos en 'cantidad':", sorted(df_ventas["cantidad"].dropna().unique()))

### 2.1 Eliminar duplicados exactos

In [ ]:
df = df_ventas.drop_duplicates().copy()
print(f"Filas después de quitar duplicados: {len(df)} (se eliminaron {len(df_ventas) - len(df)})")

### 2.2 Tratar valores nulos

Regla de negocio del ejercicio: si no sabemos la **tienda** o el **producto**, no podemos usar ese registro para el reporte, así que se descarta. El precio, si falta, se completa más adelante con la mediana del producto.

In [ ]:
antes = len(df)
df = df.dropna(subset=["id_tienda", "producto"])
print(f"Filas eliminadas por nulos críticos (tienda o producto): {antes - len(df)}")

# id_tienda venía como float por los NaN originales; ahora que no hay nulos, la pasamos a entero
df["id_tienda"] = df["id_tienda"].astype(int)

### 2.3 Estandarizar texto

Los mismos productos llegaban escritos de formas distintas (`'MOUSE'`, `'Mouse'`, `'teclado '`, etc.). Los normalizamos.

In [ ]:
df["producto"] = df["producto"].str.strip().str.lower().str.capitalize()
print(df["producto"].unique())

### 2.4 Corregir cantidades inválidas

Una cantidad de `0` o negativa no tiene sentido en una venta real: son errores de captura, así que se eliminan.

In [ ]:
antes = len(df)
df = df[df["cantidad"] > 0]
print(f"Filas eliminadas por cantidad inválida: {antes - len(df)}")

### 2.5 Imputar precios faltantes

Completamos el `precio_unitario` faltante con la mediana del **mismo producto** (una imputación más razonable que usar la mediana global).

In [ ]:
df["precio_unitario"] = df.groupby("producto")["precio_unitario"].transform(
    lambda x: x.fillna(x.median())
)
print("Nulos restantes en precio_unitario:", df["precio_unitario"].isna().sum())

### 2.6 Enriquecer los datos

Calculamos el total de cada venta y unimos (`merge`) con el catálogo de tiendas para tener el nombre y la región.

In [ ]:
df["total_venta"] = df["cantidad"] * df["precio_unitario"]

df = df.merge(df_tiendas, on="id_tienda", how="left")
df.head()

### 2.7 Agregar (resumen para el reporte final)

In [ ]:
resumen_por_tienda = (
    df.groupby(["nombre_tienda", "region"])
    .agg(
        total_transacciones=("id_venta", "count"),
        unidades_vendidas=("cantidad", "sum"),
        ingresos_totales=("total_venta", "sum"),
    )
    .reset_index()
    .sort_values("ingresos_totales", ascending=False)
)
resumen_por_tienda

---
## 3️⃣ LOAD — Carga de datos

Cargamos los datos limpios en dos formatos, como se haría en un pipeline real:

1. **SQLite**, simulando un *data warehouse*, para que otros equipos puedan consultarlo con SQL.
2. **Archivos** (CSV / Parquet), para compartir o alimentar otros procesos batch.

In [ ]:
conn = sqlite3.connect("data_warehouse.db")

# Tabla de detalle (staging limpio)
df.to_sql("ventas_limpias", conn, if_exists="replace", index=False)

# Tabla agregada (lista para reporting)
resumen_por_tienda.to_sql("resumen_ventas_tienda", conn, if_exists="replace", index=False)

print("Datos cargados en 'data_warehouse.db' ✅")

### 3.1 Verificamos con una consulta SQL

In [ ]:
query = """
SELECT region, SUM(ingresos_totales) AS ingresos_region
FROM resumen_ventas_tienda
GROUP BY region
ORDER BY ingresos_region DESC
"""
pd.read_sql(query, conn)

### 3.2 Exportamos también a CSV y Parquet

In [ ]:
df.to_csv("ventas_limpias.csv", index=False)
resumen_por_tienda.to_parquet("resumen_ventas_tienda.parquet", index=False)

conn.close()
print("Archivos exportados: ventas_limpias.csv, resumen_ventas_tienda.parquet")

---
## ✅ Resumen del pipeline

| Etapa | Qué hicimos |
|---|---|
| **Extract** | Leímos ventas "sucias" desde un CSV + catálogo de tiendas desde otra fuente |
| **Transform** | Quitamos duplicados, tratamos nulos, corregimos tipos y valores inválidos, estandarizamos texto, enriquecimos con `merge` y agregamos con `groupby` |
| **Load** | Guardamos en SQLite (data warehouse) y exportamos a CSV/Parquet |

## 📝 Ejercicios para los estudiantes

1. Agrega una nueva regla de limpieza: descarta ventas con `precio_unitario` mayor a `1.000.000` (posible error de digitación).
2. Crea una nueva tabla agregada `resumen_por_producto` con el total vendido por producto.
3. Modifica el pipeline para que registre en un log (una lista, o un archivo `.txt`) cuántas filas se eliminaron en cada paso de limpieza.
4. **(Reto)** Reescribe este notebook como un script `.py` con funciones separadas `extract()`, `transform()` y `load()`, y una función `main()` que las orqueste.
5. **(Reto)** Agrega manejo de errores (`try/except`) alrededor de la carga a SQLite, registrando si algo falla.